# 03. Семантический поиск: дообучение bi-encoder

**Зачем.** EDA показал: у ~8% пар «запрос → выбранное объявление» нет общих слов даже после стемминга
(«печник» → «Установка печей», «ямобур» → «экскаватор»). 62% запросов бенчмарка не встречались в train.
Лексический поиск такие пары не найдёт — нужна модель, которая сравнивает смысл.

**Как.** Bi-encoder кодирует запрос и объявление в векторы; близость = косинус. Корпус кодируется один раз,
поиск — умножение матриц, поэтому подход годится для кандидатогенерации.

* Базовые модели (open-source; веса скачиваются с HuggingFace один раз, дальше всё локально, без внешних API):
  `intfloat/multilingual-e5-base` (MIT), `intfloat/multilingual-e5-large` (MIT), `deepvk/USER-bge-m3` (Apache-2.0,
  русскоязычная версия bge-m3). Три разные модели дают ранжировщику разные «взгляды» на смысл.
* Данные: 214 тыс. пар (запрос + фильтры → заголовок + вид/тип услуги + услуги из прайса + начало описания) из `02_prepare`.
  Отложенные для валидации поиски в обучение не попадают.
* Функция потерь: `CachedMultipleNegativesRankingLoss` — для каждого запроса правильное объявление должно быть ближе,
  чем объявления остальных запросов батча. Батч 256 (больше негативов), считается кусками по 64 для экономии памяти.
* `NO_DUPLICATES`: в одном батче нет одинаковых текстов, иначе появились бы ложные негативы.
* Префиксы `query: ` / `passage: ` — формат, в котором обучалась e5.

Код обучения — `src/train_dense.py`. Там же после обучения считаются эмбеддинги корпуса и запросов
(`artifacts/emb_<model>_*.npy`), их использует `04_retrieve_rank`.

Обучение шло на RTX 5060 Ti 16 ГБ (bf16): e5-base — ~45 мин (2 эпохи), e5-large — ~2.3 ч (2 эпохи),
USER-bge-m3 — ~1.2 ч (1 эпоха).

In [1]:
import subprocess, sys
from pathlib import Path
ROOT = Path('..').resolve()

def run(args):
    # Запуск обучения отдельным процессом: так освобождается память GPU после завершения
    cmd = [sys.executable, '-m', 'src.train_dense'] + args
    print(' '.join(cmd))
    return subprocess.run(cmd, cwd=ROOT, check=True)

RETRAIN = False   # True — переобучить модели с нуля; False — использовать готовые эмбеддинги из artifacts/

In [2]:
if RETRAIN:
    run(['--model', 'intfloat/multilingual-e5-base', '--name', 'e5b', '--epochs', '2', '--bs', '256', '--lr', '3e-5'])
    run(['--model', 'intfloat/multilingual-e5-large', '--name', 'e5l', '--epochs', '2', '--bs', '256',
         '--mini_bs', '32', '--lr', '2e-5'])
    run(['--model', 'deepvk/USER-bge-m3', '--name', 'bgem3', '--epochs', '1', '--bs', '256', '--mini_bs', '32', '--lr', '2e-5'])
    run(['--model', 'intfloat/multilingual-e5-base', '--name', 'e5b_raw', '--encode_only'])   # исходная модель для сравнения

## Быстрая проверка: только семантический поиск

Recall@50 по косинусу без учёта локации и остальных сигналов — чтобы увидеть вклад нейросети отдельно.
Сравниваем с исходной (не дообученной) e5, если её эмбеддинги посчитаны.

In [3]:
import numpy as np, pandas as pd
sys.path.insert(0, str(ROOT))
from src.prep import recall_at_k

ART = ROOT / 'artifacts'
corpus_ids = pd.read_parquet(ART / 'corpus.parquet', columns=['item_id'])['item_id'].values
q_val = pd.read_parquet(ART / 'queries_val.parquet')

def dense_recall(name, k=50):
    E = np.load(ART / f'emb_{name}_items.npy').astype(np.float32)
    Q = np.load(ART / f'emb_{name}_q_val.npy').astype(np.float32)
    preds = []
    for b in range(0, len(Q), 512):
        S = Q[b:b + 512] @ E.T
        top = np.argpartition(-S, k, axis=1)[:, :k]
        preds += [list(corpus_ids[t]) for t in top]
    return recall_at_k(preds, q_val['relevant'].tolist(), k)

for name in ['e5b_raw', 'e5b', 'e5l', 'bgem3']:
    if (ART / f'emb_{name}_items.npy').exists():
        print(f'{name:8s} Recall@50 (только косинус, без локации): {dense_recall(name):.4f}')

e5b_raw  Recall@50 (только косинус, без локации): 0.3770


e5b      Recall@50 (только косинус, без локации): 0.4402


e5l      Recall@50 (только косинус, без локации): 0.4597


bgem3    Recall@50 (только косинус, без локации): 0.4400
